# 13. 무효전력 이상치 탐지 (Q)

## 이상치 기준
- 물리적 기준: 단방향 소비 계량기에서 Q < 0 (유도성 부하에서 무효전력은 양수가 정상)
- 통계적 기준: 계량기별 일별 평균값 기준 평균 ± 3σ 초과
- 발전/양방향 계량기는 물리적 기준 제외

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

# 발전/양방향 계량기
BIDIRECTIONAL = {
    'H1.Z20', 'H1.ZE20',
    'V.Z84', 'V.ZE84',
    'V.Z81', 'V.Z82',
    'H1.Z310', 'H2.Z311', 'H3.Z312',
    'H1.Z29', 'H1.Z28',
    'H2.T.Z33',
}

ALL_METERS = [
    'H1.Z10', 'H1.Z11', 'H1.Z12', 'H1.Z13', 'H1.Z14', 'H1.Z15', 'H1.Z16',
    'H1.Z17', 'H1.Z18', 'H1.Z19', 'H1.Z20', 'H1.Z21', 'H1.Z22', 'H1.Z23',
    'H1.Z24', 'H1.Z25', 'H1.Z26', 'H1.Z27', 'H1.Z28', 'H1.Z29', 'H1.Z310',
    'H1.ZE20', 'H2.T.Z30', 'H2.T.Z31', 'H2.T.Z32', 'H2.T.Z33', 'H2.T.Z34',
    'H2.Z311', 'H2.Z35', 'H2.Z64', 'H2.Z65', 'H2.Z66', 'H2.Z67', 'H2.Z68',
    'H2.Z69', 'H2.Z70', 'H2.ZE64', 'H2.ZE65', 'H2.ZE66', 'H2.ZE67', 'H2.ZE74',
    'H3.Z312', 'H3.Z40', 'H3.Z41', 'H3.Z42', 'H4.Z50', 'H4.Z51', 'H4.ZE50',
    'H4.ZE51', 'V.Z81', 'V.Z82', 'V.Z84', 'V.ZE84'
]

save_dir = ROOT / 'outputs/tables/anomaly'
save_dir.mkdir(parents=True, exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    if df.empty:
        return pd.DataFrame()
    df['day'] = pd.to_datetime(df['day'])
    return df


def detect_anomaly(meter, measurement):
    df = fetch_daily(meter, measurement)
    if df.empty:
        return None

    results = []

    # 1. 물리적 기준: 단방향 소비 계량기에서 Q < 0
    if meter not in BIDIRECTIONAL:
        physical = df[df['min_val'] < 0].copy()
        physical['anomaly_type'] = '물리적이상(음수무효전력)'
        physical['criterion'] = 'min_val < 0 VAR'
        if len(physical) > 0:
            results.append(physical)

    # 2. 통계적 기준: 평균 ± 3σ
    mean_val = df['avg_val'].mean()
    std_val  = df['avg_val'].std()
    if std_val == 0:
        return None
    upper = mean_val + 3 * std_val
    lower = mean_val - 3 * std_val
    stat = df[(df['avg_val'] > upper) | (df['avg_val'] < lower)].copy()
    stat['anomaly_type'] = '통계적이상(3sigma)'
    stat['criterion'] = f'mean={mean_val:.2f}, sigma={std_val:.2f}, lower={lower:.2f}, upper={upper:.2f}'
    if len(stat) > 0:
        results.append(stat)

    if not results:
        return None

    result = pd.concat(results).drop_duplicates('day').sort_values('day')
    result['meter'] = meter
    result['measurement'] = measurement
    return result[['meter', 'measurement', 'day', 'min_val', 'max_val', 'avg_val', 'anomaly_type', 'criterion']]

In [3]:
all_results = []

for meter in ALL_METERS:
    result = detect_anomaly(meter, 'Q')
    if result is not None and len(result) > 0:
        print(f'{meter} Q: {len(result)}건')
        all_results.append(result)

if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_reactive.csv', index=False)
    print(f'\n총 {len(final)}건 저장 완료')
else:
    print('이상치 없음')

/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 Q: 2191건


/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z11 Q: 37건


/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


...

H4.Z50 Q: 2건


/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H4.Z51 Q: 2000건


/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_112531/2491705871.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_112531/2491705871.py:18: UserWarning:


총 29223건 저장 완료


In [4]:
if all_results:
    summary = final.groupby(['meter', 'measurement', 'anomaly_type']).agg(
        건수=('day', 'count'),
        시작일=('day', 'min'),
        종료일=('day', 'max'),
        min_val=('min_val', 'min'),
        max_val=('max_val', 'max'),
        criterion=('criterion', 'first')
    ).reset_index()
    summary.to_csv(save_dir / 'anomaly_reactive_summary.csv', index=False)
    print(summary.to_string())

       meter measurement   anomaly_type    건수        시작일        종료일       min_val       max_val                                                     criterion
0     H1.Z10           Q  물리적이상(음수무효전력)  2189 2018-01-01 2023-12-31  -2433.708167   3289.336167                                               min_val < 0 VAR
1     H1.Z10           Q  통계적이상(3sigma)     2 2019-06-15 2019-06-16    631.861000    803.869167    mean=-1000.88, sigma=279.19, lower=-1838.46, upper=-163.30
2     H1.Z11           Q  통계적이상(3sigma)    37 2018-05-29 2023-08-22   4447.619814  57866.433470  mean=5018.82, sigma=6611.88, lower=-14816.83, upper=24854.47
3     H1.Z12           Q  물리적이상(음수무효전력)   229 2018-02-02 2023-12-15    -25.179500  38235.709833                                               min_val < 0 VAR
4     H1.Z12           Q  통계적이상(3sigma)    39 2018-05-29 2023-08-22   5718.359203  57938.452462  mean=5078.42, sigma=6702.61, lower=-15029.41, upper=25186.24
5     H1.Z13           Q  통계적이상(3sigma)    18 2018-0